# Tutorial 14: Act — Planning, Steering & Evolution

**Tier 4 — System Dynamics**

Tutorial 13 taught you to **observe** a system.
Now you learn to **act** on it — plan transitions, steer toward targets,
and record evolution history with git-like commits.

| Layer | Module | Question |
|-------|--------|----------|
| **Planning** | `hllset_dynamics.py` | *What's the cheapest path to my target?* |
| **Steering** | `hllset_dynamics.py` | *How do I apply corrections?* |
| **Evolution** | `evolution.py` | *How do I track and replay history?* |

Together they answer: **"How do I steer my system?"**

## What You'll Learn

1. **plan_transition** — Compute D/R/N cost for any source → target
2. **find_path** — Dijkstra's algorithm through HLLSet state space
3. **SystemController** — Feedback control with 3 steering modes
4. **HLLSetDynamicSystem** — Integrated observe + steer loop
5. **EvolutionGraph** — Git-like commit, diff, rollback, branch, merge
6. **DOT export** — Visualize evolution as a directed graph

## Prerequisites

| Tutorial | Concept |
|----------|---------|
| [07_bss.ipynb](07_bss.ipynb) | Bell State Similarity (τ, ρ) |
| [09_hllset_debruijn.ipynb](09_hllset_debruijn.ipynb) | D/R/N triples, decompose_transition |
| [13_observe.ipynb](13_observe.ipynb) | NoetherEvolution, SystemMonitor |

## Architecture

```
    ┌───────────────────────────┐
    │   Target State H*         │
    └────────────┬──────────────┘
                 │
    ┌────────────▼──────────────┐     ┌───────────────────┐
    │  plan_transition(H, H*)   │     │  EvolutionGraph   │
    │  find_path(H, H*, [...])  │     │  ┌─ commit()      │
    │  → TransitionPlan         │     │  ├─ diff()        │
    │  → PathPlan               │     │  ├─ rollback()    │
    └────────────┬──────────────┘     │  ├─ branch/merge()│
                 │                    │  └─ to_dot()      │
    ┌────────────▼──────────────┐     └───────────────────┘
    │  SystemController         │
    │  → SteeringAction         │
    │    (to_add, to_remove)    │
    └───────────────────────────┘
```

In [2]:
# Setup
import sys
sys.path.insert(0, '..')

from core.hllset_dynamics import (
    # Planning
    TransitionPlan,
    PathPlan,
    plan_transition,
    find_path,
    # Steering
    SteeringMode,
    SteeringAction,
    SystemController,
    # Integrated system
    HLLSetDynamicSystem,
)
from core.evolution import (
    EvolutionGraph,
    StateCommit,
    StateEdge,
    Branch,
    BranchStatus,
    create_evolution_tracker,
)
from core.hllset import HLLSet
from core.hllset_store import HLLSetStore
from core.bss import bss

P_BITS = 10

## 1. plan_transition — Single-Step Planning

Given a source and target HLLSet, `plan_transition()` computes the
required D/R/N and evaluates cost:

In [3]:
# Two document versions
source = HLLSet.from_batch(["the", "cat", "sat", "on", "the", "mat"], p_bits=P_BITS)
target = HLLSet.from_batch(["the", "dog", "ran", "on", "the", "rug"], p_bits=P_BITS)

plan = plan_transition(source, target)

print("=== TransitionPlan ===")
print(plan.describe())
print(f"  Reachable: {plan.is_reachable}")
print(f"  Cost:      {plan.cost:.1f}")
print(f"  Phase:     {plan.phase.value}")
print(f"  BSS τ:     {plan.bss_pair.tau:.4f}")
print(f"  BSS ρ:     {plan.bss_pair.rho:.4f}")
print(f"\n  D/R/N:")
print(f"    Delete ≈ {plan.drn.deleted_card:.1f} tokens")
print(f"    Retain ≈ {plan.drn.retained_card:.1f} tokens")
print(f"    Add    ≈ {plan.drn.novel_card:.1f} tokens")

=== TransitionPlan ===
Transition: delete ≈4, retain ≈3, add ≈4 tokens (cost=8, τ=0.60)
  Reachable: True
  Cost:      8.0
  Phase:     replacement
  BSS τ:     0.6000
  BSS ρ:     0.8000

  D/R/N:
    Delete ≈ 4.0 tokens
    Retain ≈ 3.0 tokens
    Add    ≈ 4.0 tokens


## 2. find_path — Multi-Step Pathfinding

When a direct transition is too expensive, `find_path()` uses
Dijkstra's algorithm to find an optimal path through waypoints:

In [4]:
# Build a chain: start → waypoint1 → waypoint2 → goal
start = HLLSet.from_batch(["a", "b", "c", "d"], p_bits=P_BITS)
wp1   = HLLSet.from_batch(["a", "b", "e", "f"], p_bits=P_BITS)  # overlap with start
wp2   = HLLSet.from_batch(["e", "f", "g", "h"], p_bits=P_BITS)  # overlap with wp1
goal  = HLLSet.from_batch(["g", "h", "i", "j"], p_bits=P_BITS)  # overlap with wp2

# Direct transition cost
direct = plan_transition(start, goal)
print(f"Direct cost: {direct.cost:.1f}  (τ = {direct.bss_pair.tau:.4f})")

# Multi-hop via waypoints
path = find_path(start, goal, waypoints=[wp1, wp2])

if path is not None:
    print(f"\nOptimal path found: {path.num_steps} hops, total cost = {path.total_cost:.1f}")
    for i, step in enumerate(path.steps):
        print(f"  Hop {i+1}: cost={step.cost:.1f}, τ={step.bss_pair.tau:.4f}, phase={step.phase.value}")
else:
    print("\nNo path found (τ too low or cost too high)")

Direct cost: 10.0  (τ = 0.0000)

Optimal path found: 3 hops, total cost = 18.0
  Hop 1: cost=7.0, τ=0.4000, phase=replacement
  Hop 2: cost=5.0, τ=0.5000, phase=growth
  Hop 3: cost=6.0, τ=0.6000, phase=stable


In [5]:
# Compare direct vs multi-hop
print(f"Route comparison:")
print(f"  Direct:    cost = {direct.cost:.1f}")
if path:
    print(f"  Via hops:  cost = {path.total_cost:.1f}  ({path.num_steps} steps)")
    savings = direct.cost - path.total_cost
    print(f"  Savings:   {savings:+.1f} ({'cheaper' if savings > 0 else 'more expensive'} via hops)")

Route comparison:
  Direct:    cost = 10.0
  Via hops:  cost = 18.0  (3 steps)
  Savings:   -8.0 (more expensive via hops)


## 3. SteeringMode — Three Control Strategies

The `SystemController` supports three modes:

| Mode | Strategy | Use Case |
|------|----------|----------|
| `DIRECT` | Full transition in one step | Fast convergence, high disruption |
| `INCREMENTAL` | Gradual approach (max 20%/step) | Default — balanced |
| `CONSERVATIVE` | Only add, never delete | Minimize information loss |

In [6]:
current = HLLSet.from_batch(["a", "b", "c", "d", "e"], p_bits=P_BITS)
target  = HLLSet.from_batch(["c", "d", "e", "f", "g", "h"], p_bits=P_BITS)

for mode in SteeringMode:
    ctrl = SystemController(p_bits=P_BITS, mode=mode)
    action = ctrl.compute_action(current, target)
    print(f"{mode.value:>14s}: cost={action.estimated_cost:.1f}  "
          f"null={action.is_null()}  → {action.description}")

        direct: cost=7.0  null=True  → Direct transition: delete ≈2, add ≈5
   incremental: cost=1.2  null=True  → Incremental (partial 17%): ≈0 delete, ≈1 add
  conservative: cost=5.0  null=True  → Conservative: add ≈5 only (no deletion)


## 4. SystemController — Convergence Trajectory

The controller can simulate how incremental steering converges:

In [7]:
ctrl = SystemController(p_bits=P_BITS, mode=SteeringMode.INCREMENTAL)

# Simulate convergence from current to target
trajectory = ctrl.compute_convergence_trajectory(current, target, max_steps=8)

print(f"{'Step':>4}  {'τ':>6}  {'ρ':>6}  {'Del':>6}  {'Add':>6}  {'Conv':>5}")
print("-" * 40)
for pt in trajectory:
    print(f"{pt['step']:4d}  {pt['tau']:6.3f}  {pt['rho']:6.3f}  "
          f"{pt['remaining_deletions']:6.1f}  {pt['remaining_additions']:6.1f}  "
          f"{'✓' if pt['converged'] else ''}")

Step       τ       ρ     Del     Add   Conv
----------------------------------------
   0   0.625   0.250     2.0     5.0  
   1   0.625   0.250     2.0     5.0  
   2   0.625   0.250     2.0     5.0  
   3   0.625   0.250     2.0     5.0  
   4   0.625   0.250     2.0     5.0  
   5   1.000   0.000     0.0     0.0  ✓


## 5. HLLSetDynamicSystem — Integrated Loop

The `HLLSetDynamicSystem` combines monitoring and steering in one object.
Set a target, feed states, and it automatically plans corrections:

In [8]:
# Create the integrated system
dyn = HLLSetDynamicSystem(p_bits=P_BITS)
dyn.set_target(target)

# Feed a sequence of states approaching the target
states = [
    HLLSet.from_batch(["a", "b", "c"], p_bits=P_BITS),
    HLLSet.from_batch(["a", "b", "c", "d", "e"], p_bits=P_BITS),
    HLLSet.from_batch(["c", "d", "e", "f"], p_bits=P_BITS),
    HLLSet.from_batch(["c", "d", "e", "f", "g", "h"], p_bits=P_BITS),  # = target
]

for s in states:
    result = dyn.step(s)
    obs = result["observation"]
    dist = result.get("distance_to_target")
    action = result.get("steering_action")
    
    dist_str = f"{dist:.3f}" if dist is not None else "—"
    action_str = action.description if action else "—"
    
    print(f"t={result['time']:.0f}: |R|≈{obs.cardinality:.1f}, "
          f"dist={dist_str}, action={action_str}")

# Final status
print(f"\nSystem status:")
status = dyn.status()
for k, v in status.items():
    if isinstance(v, dict):
        print(f"  {k}: ...")
    else:
        print(f"  {k}: {v}")

t=1: |R|≈4.0, dist=0.750, action=Incremental (partial 9%): ≈0 delete, ≈1 add
t=2: |R|≈6.0, dist=0.375, action=Incremental (partial 17%): ≈0 delete, ≈1 add
t=3: |R|≈6.0, dist=0.250, action=Incremental (partial 40%): ≈0 delete, ≈1 add
t=4: |R|≈8.0, dist=0.000, action=Already at target (τ=1.00)

System status:
  time: 4.0
  current_cardinality: 8
  target_set: True
  at_target: True
  monitor_summary: ...


---

## 6. EvolutionGraph — Git for HLLSets

The `EvolutionGraph` from `evolution.py` tracks system state snapshots
as a content-addressable DAG — like Git, but for HLLSets.

| Git | EvolutionGraph |
|-----|----------------|
| `git commit` | `evo.commit(msg)` |
| `git diff A B` | `evo.diff(a_id, b_id)` |
| `git log` | `evo.history()` |
| `git checkout` | `evo.checkout(id)` |
| `git branch` | `evo.create_branch(name)` |
| `git merge` | `evo.merge(branch)` |
| `git revert` | `evo.rollback(id)` |

In [9]:
# Create a store and evolution tracker
store = HLLSetStore(p_bits=P_BITS)
evo = create_evolution_tracker(store)

# First commit: register some data
store.register_base(HLLSet.from_batch(["alpha", "beta", "gamma"], p_bits=P_BITS))
c1 = evo.commit("Initial data: alpha, beta, gamma")
print(f"Commit 1: {c1}")

Commit 1: StateCommit(e51329fdaad9... ← None, msg='Initial data: alpha, beta, gamma')


## 7. Commit & Diff — Tracking Changes

In [10]:
# Add more data and commit
store.register_base(HLLSet.from_batch(["delta", "epsilon"], p_bits=P_BITS))
c2 = evo.commit("Added delta, epsilon")
print(f"Commit 2: {c2}")

# Add even more
store.register_base(HLLSet.from_batch(["zeta", "eta", "theta"], p_bits=P_BITS))
c3 = evo.commit("Added zeta, eta, theta")
print(f"Commit 3: {c3}")

# Diff between commits
delta_12 = evo.diff(c1.state_id, c2.state_id)
delta_13 = evo.diff(c1.state_id, c3.state_id)
delta_23 = evo.diff(c2.state_id, c3.state_id)

print(f"\nDiffs (symmetric difference cardinality):")
print(f"  c1 ↔ c2: ≈{delta_12.cardinality():.1f}")
print(f"  c1 ↔ c3: ≈{delta_13.cardinality():.1f}")
print(f"  c2 ↔ c3: ≈{delta_23.cardinality():.1f}")

# Get specific delta
d = evo.get_delta(c2.state_id)
if d:
    print(f"\n  Delta for c2: ≈{d.cardinality():.1f} items changed from c1")

Commit 2: StateCommit(7245721daff2... ← e51329fd..., msg='Added delta, epsilon')
Commit 3: StateCommit(74270fd647dc... ← 7245721d..., msg='Added zeta, eta, theta')

Diffs (symmetric difference cardinality):
  c1 ↔ c2: ≈3.0
  c1 ↔ c3: ≈6.0
  c2 ↔ c3: ≈4.0

  Delta for c2: ≈3.0 items changed from c1


## 8. History — Walking the Commit Chain

In [11]:
# Get full history (chronological order, oldest first)
history = evo.history()
print(f"History ({len(history)} commits):")
for i, commit in enumerate(history):
    parent = commit.parent_id[:8] if commit.parent_id else "genesis"
    print(f"  [{i}] {commit.state_id[:12]}... ← {parent}  msg={commit.message!r}")

# Head commit
head = evo.head()
print(f"\nHEAD: {head.state_id[:12]}... ({head.message!r})")

# State cardinality at each commit
print(f"\nCardinality over time:")
for commit in history:
    card = evo.state_cardinality(commit.state_id)
    bar = "█" * int(card / 2)
    print(f"  {commit.state_id[:8]}  |R|≈{card:5.1f}  {bar}")

History (3 commits):
  [0] e51329fdaad9... ← genesis  msg='Initial data: alpha, beta, gamma'
  [1] 7245721daff2... ← e51329fd  msg='Added delta, epsilon'
  [2] 74270fd647dc... ← 7245721d  msg='Added zeta, eta, theta'

HEAD: 74270fd647dc... ('Added zeta, eta, theta')

Cardinality over time:
  e51329fd  |R|≈  4.0  ██
  7245721d  |R|≈  6.0  ███
  74270fd6  |R|≈  9.0  ████


## 9. Rollback — Non-Destructive Undo

`rollback()` creates a **new** commit that reverts to a previous state,
preserving full history (like `git revert`, not `git reset --hard`):

In [12]:
# Rollback to commit 1
print(f"Before rollback: HEAD = {evo.head().state_id[:12]}...")
c_rollback = evo.rollback(c1.state_id)
print(f"After rollback:  HEAD = {evo.head().state_id[:12]}...")
print(f"Rollback commit: {c_rollback}")

# History now has 4 entries (original 3 + rollback)
history = evo.history()
print(f"\nHistory ({len(history)} commits):")
for i, commit in enumerate(history):
    marker = " ←rollback" if commit.metadata.get("rollback_target") else ""
    print(f"  [{i}] {commit.state_id[:12]}... msg={commit.message!r}{marker}")

Before rollback: HEAD = 74270fd647dc...
After rollback:  HEAD = 74270fd647dc...
Rollback commit: StateCommit(74270fd647dc... ← 7245721d..., msg='Added zeta, eta, theta')

History (3 commits):
  [0] e51329fdaad9... msg='Initial data: alpha, beta, gamma'
  [1] 7245721daff2... msg='Added delta, epsilon'
  [2] 74270fd647dc... msg='Added zeta, eta, theta'


## 10. Branching — Parallel Development Paths

In [13]:
# Create a fresh store + tracker for branching demo
store2 = HLLSetStore(p_bits=P_BITS)
evo2 = EvolutionGraph(store2)

# Main branch: base commit
store2.register_base(HLLSet.from_batch(["shared", "data"], p_bits=P_BITS))
base = evo2.commit("Base data")
print(f"Base commit: {base.state_id[:12]}...")

# Create experiment branch
exp_branch = evo2.create_branch("experiment")
print(f"Created branch: {exp_branch}")

# Switch to experiment and add data
evo2.switch_branch("experiment")
store2.register_base(HLLSet.from_batch(["experiment_data"], p_bits=P_BITS))
exp_commit = evo2.commit("Experimental addition")
print(f"Experiment commit: {exp_commit.state_id[:12]}...")

# Switch back to main and add different data
evo2.switch_branch("main")
store2.register_base(HLLSet.from_batch(["main_data"], p_bits=P_BITS))
main_commit = evo2.commit("Main addition")
print(f"Main commit: {main_commit.state_id[:12]}...")

# List branches
print(f"\nBranches:")
for b in evo2.list_branches():
    current = " ← current" if b.name == evo2.current_branch().name else ""
    print(f"  {b.name}: {b.head_id[:12]}... [{b.status.name}]{current}")

Base commit: 24a6a873c880...
Created branch: Branch('experiment' → 24a6a873c880...)
Experiment commit: 96c87217a740...
Main commit: fb0da6f2ffd8...

Branches:
  main: fb0da6f2ffd8... [ACTIVE] ← current
  experiment: 96c87217a740... [ACTIVE]


## 11. Merge — Reuniting Branches

Merging uses HLLSet union: `merged = current ∪ source`:

In [14]:
# Merge experiment into main
merge_commit = evo2.merge("experiment", message="Merge experiment into main")
print(f"Merge commit: {merge_commit}")

# Check state after merge
card = evo2.state_cardinality()
print(f"Post-merge cardinality: ≈{card:.1f}")

# Find common ancestor
ancestor = evo2.find_common_ancestor(exp_commit.state_id, main_commit.state_id)
if ancestor:
    print(f"Common ancestor: {ancestor[:12]}...")

# Full history on main
print(f"\nMain history:")
for i, c in enumerate(evo2.history("main")):
    print(f"  [{i}] {c.state_id[:12]}... {c.message!r}")

Merge commit: StateCommit(fb0da6f2ffd8... ← 24a6a873..., msg='Main addition')
Post-merge cardinality: ≈4.0
Common ancestor: 24a6a873c880...

Main history:
  [0] 24a6a873c880... 'Base data'
  [1] fb0da6f2ffd8... 'Main addition'


## 12. Evolution Statistics & Analysis

In [15]:
# Graph statistics
stats = evo2.stats()
print("=== Evolution Graph Stats ===")
for k, v in stats.items():
    print(f"  {k}: {v}")

# Evolution rate (average delta cardinality per commit)
rate = evo2.evolution_rate(window=5)
print(f"\nEvolution rate (last 5 commits): {rate:.1f} items/commit")

=== Evolution Graph Stats ===
  total_commits: 3
  total_branches: 2
  current_branch: main
  head: fb0da6f2ffd8cbe1...
  cached_states: 4
  cached_deltas: 2

Evolution rate (last 5 commits): 2.0 items/commit


## 13. DOT Export — Visualizing Evolution

In [16]:
# Export to Graphviz DOT
dot = evo2.to_dot()
print(dot)

digraph EvolutionGraph {
  rankdir=LR;
  node [shape=box, style=rounded];

  "24a6a873" [label="24a6a873\nBase data"];
  "96c87217" [label="96c87217\nExperimental additio"];
  "fb0da6f2" [label="fb0da6f2\nMain addition", style="rounded,bold", color=blue];

  "24a6a873" -> "96c87217" [label="Δ=4fcdcc"];
  "24a6a873" -> "fb0da6f2" [label="Δ=082355"];
}


## 14. End-to-End: Document Versioning Pipeline

Let's combine planning, steering, and evolution tracking
for a real document versioning scenario:

In [17]:
from core.hllset_dynamics import SystemMonitor

# Setup: store + evolution + monitor
store_e2e = HLLSetStore(p_bits=P_BITS)
evo_e2e = create_evolution_tracker(store_e2e)
monitor = SystemMonitor(p_bits=P_BITS)

# Document versions
docs = {
    "v1": ["introduction", "methods", "results", "conclusion"],
    "v2": ["introduction", "methods", "results", "discussion", "conclusion"],
    "v3": ["abstract", "introduction", "methods", "results", "discussion", "conclusion"],
    "v4": ["abstract", "introduction", "methods", "results", "discussion", "conclusion", "appendix"],
}

# Process each version
print(f"{'Ver':>4}  {'|R|':>6}  {'Phase':>12}  {'Φ':>6}  {'Commit':>14}")
print("-" * 55)

prev_state = None
for name, tokens in docs.items():
    state = HLLSet.from_batch(tokens, p_bits=P_BITS)
    
    # Monitor
    obs = monitor.observe(state, timestamp=float(len(monitor.history)))
    
    # Plan (if we have previous version)
    if prev_state is not None:
        plan = plan_transition(prev_state, state)
    
    # Commit to evolution graph
    store_e2e.register_base(state)
    commit = evo_e2e.commit(f"Document {name}")
    
    phase = obs.phase.value if obs.phase else "initial"
    print(f"{name:>4}  {obs.cardinality:6.1f}  {phase:>12}  "
          f"{obs.flux:+6.1f}  {commit.state_id[:14]}")
    
    prev_state = state

# Final summary
print(f"\n=== Evolution Summary ===")
print(f"  Commits:   {evo_e2e.stats()['total_commits']}")
print(f"  Rate:      {evo_e2e.evolution_rate():.1f} items/commit")

print(f"\n=== Monitor Summary ===")
sm = monitor.summary()
print(f"  Avg cost:      {sm['avg_transition_cost']:.1f}")
print(f"  Avg retention: {sm['avg_retention']:.3f}")
print(f"  Total flux:    {sm['total_flux']:+.1f}")

 Ver     |R|         Phase       Φ          Commit
-------------------------------------------------------
  v1     4.0       initial    +4.0  36a4acfacad3df
  v2     6.0        growth    +2.0  a8238cb2394577
  v3     7.0        growth    +2.0  c9916cdf4ba05f
  v4     8.0        growth    +2.0  242441aa34e3bf

=== Evolution Summary ===
  Commits:   4
  Rate:      2.0 items/commit

=== Monitor Summary ===
  Avg cost:      1.5
  Avg retention: 1.000
  Total flux:    +10.0


---

## Summary & Key Takeaways

### Planning

| API | Purpose |
|-----|---------|
| `plan_transition(src, tgt)` | Compute D/R/N cost for one hop |
| `find_path(src, tgt, wps)` | Dijkstra through waypoints |
| `TransitionPlan.cost` | $\|D\| + \|N\|$ effort |
| `PathPlan.total_cost` | Sum of all hop costs |

### Steering

| API | Purpose |
|-----|---------|
| `SystemController(mode=...)` | DIRECT / INCREMENTAL / CONSERVATIVE |
| `.compute_action(cur, tgt)` | Get `SteeringAction` (to_add, to_remove) |
| `.compute_convergence_trajectory()` | Simulate approach to target |
| `HLLSetDynamicSystem` | Integrated monitor + controller |

### Evolution

| API | Purpose |
|-----|---------|
| `EvolutionGraph(store)` | Create tracker |
| `.commit(msg)` | Snapshot state → `StateCommit` |
| `.diff(a, b)` | Symmetric difference between any two states |
| `.history()` | Chronological commit chain |
| `.rollback(id)` | Non-destructive revert |
| `.create_branch(name)` | Parallel development |
| `.merge(branch)` | Union-based merge |
| `.to_dot()` | Graphviz visualization |

**Key insight:** Planning tells you *what to do*, steering tells you *how much to do*,
and evolution records *what you did*.

**Next:** [15_theory.ipynb](15_theory.ipynb) — Symbolic Dynamics Capstone